# Tools ka use model ko extra tasks karne ke liye hota hai,
# jaise database se data fetch karna, web search karna ya code run karna.
# Har tool ke 2 main parts hote hain:
# 1. Schema: Tool ka name, description aur required arguments define karta hai.
# 2. Function/Coroutine: Actual task ko execute karta hai.
# Model tool ko request kar sakta hai, lekin tool ka actual execution
# uske function ke through hota hai.

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv 

# Environment variable se Groq API key read karke set kar rahe hain
#os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")  # sirf isse v ho jara tha, load k need ni tha, jus upr ar nich waal code k
#lload k bd getenv hoa h pr .env m phle s gogle = api key h isliye bina load k v hora tha
# .env file se API key ko environment mein load karna ye v use kr skte the
load_dotenv()

# Groq ke Qwen model ko initialize kar rahe hain
model = init_chat_model("groq:openai/gpt-oss-120b")

# Prompt model ko bhej rahe hain aur response receive kar rahe hain
response = model.invoke("Why do parrots talk?")

# Model ka complete response display kar rahe hain
response


# .env file se API key ko environment mein load karna




AIMessage(content='Parrots don’t “talk” in the same way humans use language; they are exceptional mimics that can reproduce many of the sounds they hear—including human words. Their tendency to do this stems from a mix of evolutionary, anatomical, and social factors:\n\n### 1. Social nature of parrots\n- **Flock communication:** In the wild, parrots live in noisy, complex flocks where they constantly exchange calls to keep track of each other, signal danger, establish bonds, and maintain hierarchy. Being able to pick up and reproduce a wide range of vocalizations helps them stay integrated in the group.\n- **Bonding with companions:** In captivity, a human caretaker becomes the bird’s “flock mate.” Parrots often mimic the sounds that get attention, praise, or food, because those responses reinforce the behavior.\n\n### 2. Brain structure and learning ability\n- **Highly developed vocal control:** Parrots have a specialized brain region called the **“song system”** (analogous to the son

In [5]:
# binding with tools
from langchain.tools import tool
@tool #jis functn k upr likhenge it will act as decorator (act as tool)
def get_weather(location:str)->str:
    """Get the weather at the location"""# this is a docstring that play a imp role giving instrcution to llm
    return f"its summy in {location}"

model_with_tools=model.bind_tools([get_weather])
#there is also another way to bind tool with modl, in 01 while creating a model

In [6]:
# User ka question model ko bhej rahe hain
response = model_with_tools.invoke("What's the weather like in Boston?")

# Model ka response aur tool call ki details display karna
print(response)

# Response mein model ne jo tools call karne ko kaha hai, unhe access karna
for tool_call in response.tool_calls:

    # Tool ka naam print karna
    print(f"Tool: {tool_call['name']}")

    # Tool ko diye jaane wale arguments print karna
    print(f"Args: {tool_call['args']}")

# here only 1 tool call is there, so u dont need for loop, tool_call[0]['name'] ese v kr skte the

content='' additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather like in Boston?" We need to get current weather. Use function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_a23dc4ed-0380-487f-a062-8a8334237f72', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 127, 'total_tokens': 183, 'completion_time': 0.117312791, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.005757295, 'prompt_tokens_details': None, 'queue_time': 0.353947007, 'total_time': 0.123070086}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fe269835c7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a17ae5-de1c-71de-b0f7-b516fb1de772-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_a23dc4ed-0380-487f-a062-8a8334237f72

# tool execution loops

In [7]:
# STEP 1: User ka question messages list mein store karna
messages = [
    {"role": "user", "content": "What's the weather in Boston?"}
]

# Output:
# [{'role': 'user', 'content': "What's the weather in Boston?"}]


# STEP 2: Question model ko bhejna
ai_msg = model_with_tools.invoke(messages)

# Model decide karega ki weather tool use karna hai.
# Is example mein model get_weather tool call karne ki request deta hai.
# content empty ho sakta hai, kyunki abhi normal text answer nahi diya.
# tool_calls mein tool ka naam aur arguments hote hain.

# Example response:
# AIMessage(
#     content='',
#     tool_calls=[{
#         'name': 'get_weather',
#         'args': {'location': 'Boston'},
#         'id': 'call_123'
#     }]
# )


# STEP 3: Model ka response conversation mein add karna
messages.append(ai_msg)

# Output:
# Ab messages mein user ka question aur model ka tool-call response hai.


# STEP 4: Model ke requested tool ko execute karna
for tool_call in ai_msg.tool_calls:

    # Actual weather tool ko call karke data lena
    tool_result = get_weather.invoke(tool_call)

    # Example tool result:
    # ToolMessage(content='72°F, sunny', ...)

    # Tool ka result conversation mein add karna
    messages.append(tool_result)

# Output:
# Ab messages mein user ka question, model ka tool-call response
# aur tool se mila weather data hai.


# STEP 5: Updated conversation dobara model ko dena
final_response = model_with_tools.invoke(messages)

# Model ab tool ke result ko use karke final answer banayega.

# STEP 6: Final answer print karna
print(final_response.content)

# Example output:
# The current weather in Boston is 72°F and sunny.

The current weather in Boston is **sunny**. Enjoy the nice weather!


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_83180d10-99d2-48c1-bd8d-f7c65acf4fda', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 126, 'total_tokens': 165, 'completion_time': 0.081826975, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.004717872, 'prompt_tokens_details': None, 'queue_time': 0.399074593, 'total_time': 0.086544847}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17b14-2c44-7676-82ca-6ad4258d3512-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_83180d10-99d2-48c1-bd8d-f7c65acf4fda', '